# 모델 저장과 로드

학습을 한 번 끝낼 때마다 항상 다시 처음부터 학습시키는 것은 비효율적이다.

보통 저장이 필요한 이유는 다음과 같다.

1. 학습된 모델을 나중에 다시 사용하기 위해
2. 다른 사람이나 다른 환경으로 모델을 전달하기 위해
3. 서비스에서 추론용으로 불러오기 위해
4. 학습 도중 중단되었을 때 이어서 학습하기 위해

즉 저장은 단순 백업이 아니라,
**모델을 재사용 가능한 상태로 남겨두는 작업** 이라고 보면 된다.

In [32]:
import os
import random
import pickle
import joblib
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

## 1. scikit-learn / 일반 Python 객체

1. `pickle`
   - Python 표준 라이브러리이다.
   - 별도 설치 없이 바로 사용할 수 있다.
   - 범용성이 좋고, Python 객체 저장의 기본 개념을 설명하기 좋다.

2. `joblib`
   - 큰 numpy 배열이 포함된 객체를 다룰 때 자주 사용된다.
   - scikit-learn 예제나 실무 코드에서 자주 보인다.
   - 모델 + 전처리 객체 저장 예시에서 많이 쓰인다.

### 주요 메서드

1. `joblib.dump(obj, path)`
   - Python 객체를 파일로 저장한다.
   - numpy 배열처럼 큰 데이터가 들어 있는 객체를 저장할 때 자주 쓴다.

2. `joblib.load(path)`
   - `joblib.dump()`로 저장한 객체를 다시 불러온다.

3. `pickle.dump(obj, file)`
   - Python 기본 직렬화(serialization) 방식으로 객체를 파일에 저장한다.

4. `pickle.load(file)`
   - `pickle.dump()`로 저장한 객체를 다시 불러온다.

### 확장자

- `pickle`은 보통 `.pkl` 확장자를 많이 쓴다.
- `joblib`은 보통 `.joblib` 확장자를 많이 쓴다.
- 하지만 확장자는 관례일 뿐이며, 중요한 것은 내부에 무엇을 어떤 방식으로 저장했는가이다.

### 데이터 준비

In [33]:
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print('train:', X_train.shape, y_train.shape)
print('test :', X_test.shape, y_test.shape)

train: (455, 30) (455,)
test : (114, 30) (114,)


In [34]:
# 전처리와 모델을 Pipeline으로 묶는다
ml_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=SEED))
])

ml_pipeline.fit(X_train, y_train)
ml_preds = ml_pipeline.predict(X_test)
ml_acc = accuracy_score(y_test, ml_preds)
print(f'pipeline test acc : {ml_acc:.4f}')

pipeline test acc : 0.9825


### pickle(pkl)로 저장/로드

In [35]:
pickle_path = 'ml_pipeline.pkl'

# wb : write binary
with open(pickle_path, 'wb') as f:
    pickle.dump(ml_pipeline, f)
print('saved with pickle : ', pickle_path)

# rb : read binary
with open(pickle_path, 'rb') as f:
    loaded_pipeline_pickle = pickle.load(f)

loaded_preds_pickle = loaded_pipeline_pickle.predict(X_test)
loaded_acc_pickle = accuracy_score(y_test, loaded_preds_pickle)
print(f'loaded pipeline acc (pickle) : {loaded_acc_pickle:.4f}')

saved with pickle :  ml_pipeline.pkl
loaded pipeline acc (pickle) : 0.9825


### joblib으로 저장/로드

In [36]:
joblib_path = 'ml_pipeline.joblib'

# pipeline 전체를 파일로 저장
joblib.dump(ml_pipeline, joblib_path)
print('saved with joblib:', joblib_path)

# 저장했던 pipeline을 다시 불러온다
loaded_pipeline_joblib = joblib.load(joblib_path)
loaded_preds_joblib = loaded_pipeline_joblib.predict(X_test)
loaded_acc_joblib = accuracy_score(y_test, loaded_preds_joblib)
print(f'loaded pipeline acc (joblib) : {loaded_acc_joblib:.4f}')

saved with joblib: ml_pipeline.joblib
loaded pipeline acc (joblib) : 0.9825


## 2. PyTorch 

### 주요 메서드

1. `model.state_dict()`
   - 모델 안의 학습된 파라미터(weight, bias 등)를 딕셔너리 형태로 꺼낸다.
   - 모델의 학습된 값 목록이라고 생각하면 된다.

2. `model.load_state_dict(state)`
   - 저장해 둔 파라미터 딕셔너리를 현재 모델 객체에 다시 넣는다.
   - 이때 모델 구조는 미리 같은 형태로 만들어져 있어야 한다.

3. `optimizer.state_dict()`
   - optimizer 내부 상태를 딕셔너리로 꺼낸다.
   - Adam이라면 모멘텀 관련 내부 상태도 포함된다.

4. `optimizer.load_state_dict(state)`
   - 저장했던 optimizer 상태를 다시 복원한다.
   - 학습 재개용 checkpoint에서 중요하다.

5. `torch.save(obj, path)`
   - PyTorch 객체를 파일로 저장한다.
   - 보통 `state_dict()` 결과나 checkpoint 딕셔너리를 저장할 때 쓴다.

6. `torch.load(path)`
   - `torch.save()`로 저장한 객체를 다시 불러온다.

### 데이터 준비

In [37]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

dl_scaler = StandardScaler()
X_train_scaled = dl_scaler.fit_transform(X_train)
X_val_scaled = dl_scaler.transform(X_val)
X_test_scaled = dl_scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.reshape(-1, 1), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32)

In [38]:
class BinaryMLP(nn.Module):
    def __init__(self, input_dim=30, hidden_dims=[64, 32], dropout_rate=0.2):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


def evaluate_binary(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            total_loss += loss.item() * X_batch.size(0)
            all_preds.append(preds)
            all_targets.append(y_batch)

    y_pred = torch.cat(all_preds).numpy().ravel()
    y_true = torch.cat(all_targets).numpy().ravel()
    acc = accuracy_score(y_true, y_pred)
    return total_loss / len(loader.dataset), acc

In [39]:
set_seed(SEED)

model_config = {
    'input_dim': 30,
    'hidden_dims': [64, 32],
    'dropout_rate': 0.2
}

optimizer_config = {
    'optimizer_name': 'Adam',
    'lr': 0.001
}

model = BinaryMLP(**model_config)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=optimizer_config['lr'])

best_val_loss = float('inf')
best_epoch = 0
best_state = None

for epoch in range(1, 11):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate_binary(model, val_loader, criterion)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

model.load_state_dict(best_state)
test_loss, test_acc = evaluate_binary(model, test_loader, criterion)
print(f'best epoch: {best_epoch}')
print(f'test acc : {test_acc:.4f}')

epoch  1 | train_loss=0.6648 | val_loss=0.6041 | val_acc=0.7882
epoch  2 | train_loss=0.5443 | val_loss=0.4722 | val_acc=0.9412
epoch  3 | train_loss=0.3872 | val_loss=0.3153 | val_acc=0.9412
epoch  4 | train_loss=0.2485 | val_loss=0.1987 | val_acc=0.9529
epoch  5 | train_loss=0.1611 | val_loss=0.1358 | val_acc=0.9647
epoch  6 | train_loss=0.1325 | val_loss=0.1054 | val_acc=0.9765
epoch  7 | train_loss=0.1028 | val_loss=0.0853 | val_acc=0.9765
epoch  8 | train_loss=0.0865 | val_loss=0.0755 | val_acc=0.9882
epoch  9 | train_loss=0.0822 | val_loss=0.0689 | val_acc=0.9882
epoch 10 | train_loss=0.0679 | val_loss=0.0624 | val_acc=0.9882
best epoch: 10
test acc : 0.9767


### 가중치만 저장하기
주어진 파일명으로 가중치만 저장 한 뒤 다시 불러와서 test 데이터로 평가한 정확도 출력하기

In [40]:
weights_path = 'dl_binary_mlp_weights.pth'

# 1) 가중치 딕셔너리를 파일로 저장
torch.save(model.state_dict(), weights_path)
print('saved weights:', weights_path)

# 2) 같은 구조의 모델 객체를 다시 만든다.
loaded_model = BinaryMLP(**model_config)

# 3) torch.load()로 저장한 딕셔너리를 읽고, load_state_dict()로 모델에 학습된 파라미터를 채워 넣는다.
loaded_model.load_state_dict(
    torch.load(weights_path, weights_only=True)
)

# 4) 추론/평가용이므로 eval() 호출
loaded_model.eval()

# 5) test 데이터로 평가 함수 호출 후 정확도 출력
loaded_test_loss, loaded_test_acc = evaluate_binary(loaded_model, test_loader, criterion)
print(f'loaded model test acc: {loaded_test_acc:.4f}')

saved weights: dl_binary_mlp_weights.pth
loaded model test acc: 0.9767


### checkpoint 저장 : 모델 추가 정보까지 함께 저장하기
모델 설정, 옵티마이저설정, 실제 학습 상태 등을 함께 저장한 뒤 다시 불러와서 3 epoch 학습 재개하고 출력하기

In [41]:
checkpoint_path = 'dl_training_checkpoint.tar'

# 1) checkpoint에 저장할 정보를 딕셔너리로 묶는다.
checkpoint = {
    'epoch': best_epoch,
    'best_val_loss': best_val_loss,

    # 복원에 필요한 설정 정보
    'model_config': model_config,
    'optimizer_config': optimizer_config,

    # 실제 학습 상태
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}

# 2) checkpoint 딕셔너리를 파일로 저장한다.
torch.save(checkpoint, checkpoint_path)
print('saved checkpoint:', checkpoint_path)

# 3) 저장한 checkpoint 파일을 다시 불러온다.
# 이번에는 가중치만이 아니라 학습 상태 전체를 읽어온다.
loaded_checkpoint = torch.load(checkpoint_path, weights_only=False)

# 4) checkpoint 안에 저장해둔 모델 설정과 optimizer 설정을 꺼낸다.
loaded_model_config = loaded_checkpoint['model_config']
loaded_optimizer_config = loaded_checkpoint['optimizer_config']

# 5) 저장 당시와 같은 구조의 모델 객체를 다시 만든다.
new_model = BinaryMLP(**loaded_model_config)

# 6) 저장 당시 사용한 optimizer 설정에 맞춰 optimizer 객체를 다시 만든다.
if loaded_optimizer_config['optimizer_name'] == 'Adam':
    new_optimizer = optim.Adam(new_model.parameters(), lr=loaded_optimizer_config['lr'])
else:
    raise ValueError('지원하지 않는 optimizer입니다.')

# 7) 저장된 모델 가중치를 새 모델 객체에 복원한다.
new_model.load_state_dict(loaded_checkpoint['model_state_dict'])

# 8) 저장된 optimizer 상태도 새 optimizer 객체에 복원한다.
# 이렇게 해야 learning rate 상태나 momentum 같은 정보까지 이어서 사용할 수 있다.
new_optimizer.load_state_dict(loaded_checkpoint['optimizer_state_dict'])

# 9) 몇 epoch까지 학습했었는지, 최고 성능이 어땠는지도 함께 복원한다.
loaded_epoch = loaded_checkpoint['epoch']
loaded_best_val_loss = loaded_checkpoint['best_val_loss']
start_epoch = loaded_epoch + 1

# 10) 복원된 정보를 출력해서 정상적으로 불러왔는지 확인한다.
print('loaded epoch        :', loaded_epoch)
print('loaded best val loss:', round(loaded_best_val_loss, 4))
print('model config        :', loaded_model_config)
print('optimizer config    :', loaded_optimizer_config)
print('start_epoch         :', start_epoch)

# 11) 이어서 몇 epoch 더 학습해본다.
# checkpoint를 불러오면 중간부터 학습을 계속 진행할 수 있다.
for epoch in range(start_epoch, start_epoch + 3):
    train_loss = train_one_epoch(new_model, train_loader, criterion, new_optimizer)
    val_loss, val_acc = evaluate_binary(new_model, val_loader, criterion)
    print(f'epoch {epoch:2d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

saved checkpoint: dl_training_checkpoint.tar
loaded epoch        : 10
loaded best val loss: 0.0624
model config        : {'input_dim': 30, 'hidden_dims': [64, 32], 'dropout_rate': 0.2}
optimizer config    : {'optimizer_name': 'Adam', 'lr': 0.001}
start_epoch         : 11
epoch 11 | train_loss=0.0674 | val_loss=0.0590 | val_acc=0.9882
epoch 12 | train_loss=0.0675 | val_loss=0.0549 | val_acc=0.9882
epoch 13 | train_loss=0.0611 | val_loss=0.0539 | val_acc=0.9765


## 3. ONNX

ONNX(Open Neural Network Exchange)는
모델을 하나의 공통 형식으로 표현해서 다른 프레임워크나 실행 환경에서도 사용할 수 있게 돕는 형식이다.

1. PyTorch에서 모델을 학습한다.
2. 학습한 모델을 ONNX 파일로 내보낸다.
3. ONNX Runtime 같은 별도 실행 환경에서 추론한다.

### 주요 메서드

1. `torch.onnx.export(...)`
   - PyTorch 모델을 ONNX 형식 파일로 내보낸다.
   - PyTorch 모델을 다른 실행 환경에서도 돌릴 수 있는 형태로 변환한다는 의미이다.

2. `onnxruntime.InferenceSession(...)`
   - 저장된 ONNX 파일을 로드해서 추론 세션을 만든다.

3. `session.run(...)`
   - ONNX Runtime으로 실제 예측을 수행한다.

### 사용 이유

- 학습 프레임워크와 추론 환경을 분리하고 싶을 때
- 다른 언어 / 다른 서비스 환경으로 모델을 옮기고 싶을 때
- 일관된 인터페이스로 모델을 실행하고 싶을 때

### 주의할 점

- ONNX는 학습 재개용 checkpoint와는 목적이 다르다.
- 보통 ONNX는 추론용 배포 관점에서 더 많이 본다.
- 즉 checkpoint는 학습 이어하기용, ONNX는 배포/실행 호환성 쪽에 더 가깝다.

In [42]:
# ONNX 실습을 위해 아래 패키지를 설치한다.
# onnx        : PyTorch 모델을 ONNX 형식으로 export할 때 사용
# onnxscript  : 최근 PyTorch ONNX exporter가 함께 사용하는 의존성
# onnxruntime : 저장된 ONNX 모델을 실제로 불러와 추론할 때 사용
# %conda install onnx onnxscript onnxruntime

### ONNX 내보내기

In [ ]:
onnx_path = 'dl_binary_mlp.onnx'

# 더미 입력 생성 (onnx export 시에 사용할 모델 입력 예시 입력)
# batch 크기 1개, feature 수는 input_dim으로 설정
dummy_input = torch.randn(1, model_config['input_dim'])

import onnx

# export 전, 추론 전에는 평가모드로 두는 것이 안전하다 
model.eval()

# PyTorch 모델을 ONNX 형식으로 내보낸다
torch.onnx.export(
    model,                                  # ONNX로 변환할 PyTorch 모델
    dummy_input,                            # 모델 입력 예시(입력 shape 추론하는데 사용)
    onnx_path,                              # 저장할 onnx 파일 경로
    export_params=True,                     # 학습 된 weights, bias 같은 파라미터도 함께 저장
    # opset_version=12,                     
    opset_version=18,                       # ONNX 연산자(opset) 버전 지정
    do_constant_folding=True,               # 계산 결과가 고정 되는 부분은 미리 단순화 해서 저장
    input_names=['input'],                  # ONNX 그래프에서 입력 이름을 input으로 지정
    output_names=['logits'],                # ONNX 그래프에서 출력 이름을 logits으로 지정

    # dynamic_axes={
    #     # 추론 시 batch 크기가 어떤 값이 되더라도 유연하게 받을 수 있도록 설정
    #     # 입력의 0번 축은 batch 축이다.
    #     'input' : {0: 'batch_size'},    
    #     'logits' : {0: 'batch_size'}
    # }

    # 최신 ONNX exporter(dynamo=True) 방식으로 모델을 내보낸다.
    # dynamic_shapes는 입력의 동적 shape를 지정하는 옵션이다.
    # 따라서 forward(self, x)에서 입력 인자인 x만 적으면 된다.
    # 출력 logits의 batch 축은 입력 x의 batch 축을 따라가므로 따로 적지 않는다.
    dynamic_shapes={
        'x': {0: torch.export.Dim('batch_size')}
    },                                                                            
    dynamo=True
)

print('exported onnx:', onnx_path)

# https://docs.pytorch.org/docs/stable/onnx.html

[torch.onnx] Obtain model graph for `BinaryMLP([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BinaryMLP([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
exported onnx: dl_binary_mlp.onnx


c:\Users\Playdata\AppData\Local\miniforge3\envs\ai_basic_env\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


### ONNX Runtime으로 추론하기

In [46]:
# 저장 된 onnx 파일을 실제로 로드해서 추론할 때 사용한다.
import onnxruntime as ort 

# onnx 파일을 기반으로 추론 세션 생성
session = ort.InferenceSession(onnx_path)

# 테스트 데이터 중 5개의 샘플을 가져온다.
sample_input = X_test_tensor[:5].numpy().astype(np.float32)

# onnx runtime으로 실제 예측을 수행한다.
ort_outputs = session.run(
    None,
    {   
        # export 할 때 지정했던 이름으로 사용
        'input' : sample_input
    }
)

# session.run() 의 결과는 리스트 형태로 반환 된다.
# 해당 모델은 출력 값이 1개 이므로 첫 번째 결과를 꺼낸다.
ort_logits = ort_outputs[0]

# 현재 모델의 출력은 logits이므로 이진 분류로 변경해서 출력 확인
ort_probs = 1 / (1 + np.exp(-ort_logits))
ort_preds = (ort_probs >= 0.5).astype(np.float32)

print('onnx logits shape : ', ort_logits.shape)
print('onnx probs : ', np.round(ort_probs.ravel(), 4))
print('onnx preds : ', ort_preds.ravel().astype(int))

onnx logits shape :  (5, 1)
onnx probs :  [9.947e-01 4.000e-04 9.000e-04 0.000e+00 6.266e-01]
onnx preds :  [1 0 0 0 1]
